# Food101 Dataset - Exploratory Data Analysis (EDA)

This notebook performs comprehensive exploratory data analysis on the Food101 dataset, focusing on the selected subset of classes.

**Topics Covered:**
- Data distribution and class balance analysis
- Image characteristics (size, aspect ratio)
- Visual sample exploration
- Split composition and characteristics
- Class-wise statistics and patterns


In [1]:
from pathlib import Path

req_file = (Path.cwd() / "../../requirements.txt").resolve()
print("Installing from:", req_file)

%pip install -r {req_file}


Installing from: /Users/khoatran/Developer/learning-so-deep/assignments/assignment1/requirements.txt

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Enable inline matplotlib for Jupyter notebooks
%matplotlib inline

# Add project root to path
PROJECT_ROOT = Path.cwd().parent.resolve()   # .../multimodal
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from models.food101_experiment import ExperimentConfig, load_food101_datasets, humanize_class_name

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10


/Users/khoatran/Developer/learning-so-deep/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Define experiment configuration with selected Food101 classes
config = ExperimentConfig(
    model_id="openai/clip-vit-base-patch32",
    top_k=10,
    class_names=[
        "apple_pie",
        "bibimbap",
        "chicken_wings",
        "donuts",
        "eggs_benedict",
        "french_fries",
        "grilled_cheese_sandwich",
        "hamburger",
        "ice_cream",
        "pizza",
    ],
    shots_per_class=8,
    val_per_class=20,
)

print("Configuration:")
print(f"  Model: {config.model_id}")
print(f"  Dataset: {config.dataset_name}")
print(f"  Number of classes: {len(config.class_names)}")
print(f"  Classes: {[humanize_class_name(name) for name in config.class_names]}")
print(f"  Shots per class: {config.shots_per_class}")
print(f"  Validation samples per class: {config.val_per_class}")


Configuration:
  Model: openai/clip-vit-base-patch32
  Dataset: ethz/food101
  Number of classes: 10
  Classes: ['apple pie', 'bibimbap', 'chicken wings', 'donuts', 'eggs benedict', 'french fries', 'grilled cheese sandwich', 'hamburger', 'ice cream', 'pizza']
  Shots per class: 8
  Validation samples per class: 20


In [4]:
# Load Food101 datasets with selected classes
datasets_by_split, class_names, dataset_summary = load_food101_datasets(config)

print("\n" + "="*60)
print("DATASET SUMMARY")
print("="*60)
for key, value in dataset_summary.items():
    if key == "selected_class_names_readable":
        print(f"{key}:\n  {value}")
    else:
        print(f"{key}: {value}")

print("\nDataset Splits:")
for split_name, dataset in datasets_by_split.items():
    print(f"  {split_name}: {len(dataset)} samples")



DATASET SUMMARY
dataset_name: ethz/food101
available_splits: ['train', 'validation']
selected_class_names: ['apple_pie', 'bibimbap', 'chicken_wings', 'donuts', 'eggs_benedict', 'french_fries', 'grilled_cheese_sandwich', 'hamburger', 'ice_cream', 'pizza']
selected_class_names_readable:
  ['apple pie', 'bibimbap', 'chicken wings', 'donuts', 'eggs benedict', 'french fries', 'grilled cheese sandwich', 'hamburger', 'ice cream', 'pizza']
shots_per_class: 8
dev_per_class: 20
few_shot_train_size: 80
few_shot_dev_size: 200
held_out_validation_size: 2500
few_shot_train_size_per_class: 8
few_shot_dev_size_per_class: 20
evaluation_split_name: validation

Dataset Splits:
  few_shot_train: 80 samples
  few_shot_dev: 200 samples
  validation: 2500 samples


In [5]:
# Create output directory for saving plots
OUTPUT_DIR = Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"✓ Output directory created at: {OUTPUT_DIR}")
print(f"  All plots will be saved here")


✓ Output directory created at: /Users/khoatran/Developer/learning-so-deep/assignments/assignment1/multimodal/notebooks/outputs
  All plots will be saved here


## 1. Class Distribution Analysis


In [6]:
# Analyze class distribution across splits
distribution_data = []

for split_name, dataset in datasets_by_split.items():
    # Count samples per class
    label_counts = Counter()
    for sample in dataset:
        label_counts[sample['label']] += 1
    
    # Create dataframe rows
    for class_id in range(len(class_names)):
        count = label_counts[class_id]
        distribution_data.append({
            'Split': split_name.replace('_', ' ').title(),
            'Class': humanize_class_name(class_names[class_id]),
            'Count': count,
        })

dist_df = pd.DataFrame(distribution_data)

# Create pivot table for summary
pivot_dist = dist_df.pivot(index='Class', columns='Split', values='Count').fillna(0).astype(int)
print("\nClass Distribution Across Splits:")
print(pivot_dist)
print(f"\nTotal samples: {pivot_dist.sum().sum()}")


/Users/khoatran/Developer/learning-so-deep/venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))



Class Distribution Across Splits:
Split                    Few Shot Dev  Few Shot Train  Validation
Class                                                            
apple pie                          20               8         250
bibimbap                           20               8         250
chicken wings                      20               8         250
donuts                             20               8         250
eggs benedict                      20               8         250
french fries                       20               8         250
grilled cheese sandwich            20               8         250
hamburger                          20               8         250
ice cream                          20               8         250
pizza                              20               8         250

Total samples: 2780


In [7]:
# Stacked bar chart showing distribution across splits
fig, ax = plt.subplots(figsize=(14, 6))
pivot_dist.plot(kind='bar', stacked=False, ax=ax, width=0.8)
ax.set_title('Food101 Class Distribution Across Dataset Splits', fontsize=14, fontweight='bold')
ax.set_xlabel('Food Class', fontsize=12)
ax.set_ylabel('Number of Samples', fontsize=12)
ax.legend(title='Split', loc='upper right', framealpha=0.9)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_class_distribution_stacked.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Stacked distribution chart created and saved")


✓ Stacked distribution chart created and saved


/var/folders/jr/_vnkd1dx5hz54ytwj94g23sr0000gn/T/ipykernel_93645/1388003686.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Grouped bar chart for each split
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
split_names = list(datasets_by_split.keys())

for idx, split_name in enumerate(split_names):
    split_dist = dist_df[dist_df['Split'] == split_name.replace('_', ' ').title()].sort_values('Count', ascending=False)
    
    colors = sns.color_palette("husl", len(split_dist))
    axes[idx].barh(split_dist['Class'], split_dist['Count'], color=colors)
    axes[idx].set_title(f'{split_name.replace("_", " ").title()} Distribution', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Count', fontsize=11)
    axes[idx].grid(axis='x', alpha=0.3)
    
    # Add value labels
    for i, v in enumerate(split_dist['Count']):
        axes[idx].text(v + 0.5, i, str(int(v)), va='center', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_class_distribution_split_wise.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Split-wise distribution charts created and saved")


/var/folders/jr/_vnkd1dx5hz54ytwj94g23sr0000gn/T/ipykernel_93645/1397148725.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


✓ Split-wise distribution charts created and saved


## 2. Class Balance and Imbalance Analysis


In [9]:
# Calculate imbalance metrics for each split
imbalance_metrics = []

for split_name, dataset in datasets_by_split.items():
    counts = []
    for class_id in range(len(class_names)):
        count = sum(1 for sample in dataset if sample['label'] == class_id)
        counts.append(count)
    
    counts = np.array(counts)
    min_count = counts.min()
    max_count = counts.max()
    mean_count = counts.mean()
    std_count = counts.std()
    imbalance_ratio = (max_count - min_count) / max_count if max_count > 0 else 0
    
    imbalance_metrics.append({
        'Split': split_name.replace('_', ' ').title(),
        'Min': min_count,
        'Max': max_count,
        'Mean': mean_count,
        'Std': std_count,
        'Imbalance Ratio': imbalance_ratio,
    })

imbalance_df = pd.DataFrame(imbalance_metrics)
print("\nClass Balance Metrics:")
print(imbalance_df.to_string(index=False))


/Users/khoatran/Developer/learning-so-deep/venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))



Class Balance Metrics:
         Split  Min  Max  Mean  Std  Imbalance Ratio
Few Shot Train    8    8   8.0  0.0              0.0
  Few Shot Dev   20   20  20.0  0.0              0.0
    Validation  250  250 250.0  0.0              0.0


In [10]:
# Visualization: Statistical comparison across splits
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Min-Max comparison
ax = axes[0, 0]
x = np.arange(len(split_names))
width = 0.35
ax.bar(x - width/2, imbalance_df['Min'], width, label='Min', alpha=0.8)
ax.bar(x + width/2, imbalance_df['Max'], width, label='Max', alpha=0.8)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Min vs Max Samples per Class Across Splits', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([s.replace(' ', '\n') for s in imbalance_df['Split']], fontsize=10)
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Mean with error bars
ax = axes[0, 1]
ax.bar(x, imbalance_df['Mean'], yerr=imbalance_df['Std'], capsize=5, alpha=0.8, color='steelblue')
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Mean ± Std Samples per Class', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([s.replace(' ', '\n') for s in imbalance_df['Split']], fontsize=10)
ax.grid(axis='y', alpha=0.3)

# Imbalance ratio
ax = axes[1, 0]
colors = ['red' if ratio > 0.2 else 'orange' if ratio > 0.1 else 'green' 
          for ratio in imbalance_df['Imbalance Ratio']]
ax.bar(x, imbalance_df['Imbalance Ratio'], color=colors, alpha=0.8)
ax.set_ylabel('Imbalance Ratio', fontsize=11)
ax.set_title('Class Imbalance Ratio (max-min)/max', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([s.replace(' ', '\n') for s in imbalance_df['Split']], fontsize=10)
ax.axhline(y=0.1, color='green', linestyle='--', alpha=0.5, label='Good (<0.1)')
ax.axhline(y=0.2, color='red', linestyle='--', alpha=0.5, label='Poor (>0.2)')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Summary statistics table
ax = axes[1, 1]
ax.axis('off')
table_data = imbalance_df.round(3).values.tolist()
table = ax.table(cellText=table_data, 
                colLabels=imbalance_df.columns,
                cellLoc='center',
                loc='center',
                colWidths=[0.15, 0.1, 0.1, 0.1, 0.15, 0.2])
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 2)
for i in range(len(imbalance_df.columns)):
    table[(0, i)].set_facecolor('#40466e')
    table[(0, i)].set_text_props(weight='bold', color='white')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_class_balance_imbalance_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Imbalance analysis charts created and saved")


✓ Imbalance analysis charts created and saved


/var/folders/jr/_vnkd1dx5hz54ytwj94g23sr0000gn/T/ipykernel_93645/2132642822.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Image Characteristics Analysis


In [11]:
# Analyze image characteristics - we'll use validation split for comprehensive analysis
validation_dataset = datasets_by_split['validation']

image_stats = []
sample_size = min(500, len(validation_dataset))  # Limit to 500 samples for speed
samples_per_class = sample_size // len(class_names)

print(f"Analyzing {sample_size} images ({samples_per_class} per class)...")

for class_id in range(len(class_names)):
    class_count = 0
    for idx, sample in enumerate(validation_dataset):
        if sample['label'] == class_id and class_count < samples_per_class:
            image = sample['image']
            width, height = image.size
            aspect_ratio = width / height if height > 0 else 0
            
            image_stats.append({
                'Class': humanize_class_name(class_names[class_id]),
                'Width': width,
                'Height': height,
                'Aspect Ratio': aspect_ratio,
                'Area': width * height,
            })
            class_count += 1
        
        if class_count >= samples_per_class:
            break

image_stats_df = pd.DataFrame(image_stats)
print(f"✓ Analyzed {len(image_stats_df)} images")

# Print summary statistics
print("\nImage Statistics Summary:")
summary_stats = image_stats_df.groupby('Class')[['Width', 'Height', 'Aspect Ratio', 'Area']].agg(['mean', 'min', 'max', 'std']).round(2)
display(summary_stats)


Analyzing 500 images (50 per class)...


/Users/khoatran/Developer/learning-so-deep/venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


✓ Analyzed 500 images

Image Statistics Summary:


Width                   Height                   \
                           mean  min  max    std    mean  min  max    std   
Class                                                                       
apple pie                496.52  382  512  42.35  462.40  306  512  73.05   
bibimbap                 506.88  384  512  25.34  465.44  288  512  71.27   
chicken wings            499.20  384  512  38.79  450.12  289  512  74.44   
donuts                   496.42  288  512  48.75  466.82  288  512  72.44   
eggs benedict            493.14  289  512  53.53  469.18  289  512  67.83   
french fries             488.06  289  512  57.41  475.38  306  512  63.52   
grilled cheese sandwich  492.38  306  512  50.11  476.90  288  512  65.19   
hamburger                484.36  289  512  57.28  481.84  288  512  58.83   
ice cream                475.28  289  512  64.64  483.18  306  512  59.86   
pizza                    499.14  382  512  38.91  453.88  335  512  70.12   

                        Aspect Ratio                         Area          \
                                mean   min   max   std       mean     min   
Class                                                                       
apple pie                       1.11  0.75  1.67  0.25  228823.04  156672   
bibimbap                        1.12  0.75  1.78  0.23  235683.84  147456   
chicken wings                   1.15  0.75  1.77  0.25  223907.84  147968   
donuts                          1.10  0.56  1.78  0.26  231034.88  147456   
eggs benedict                   1.08  0.56  1.77  0.24  230563.84  147968   
french fries                    1.05  0.56  1.67  0.23  231137.28  147968   
grilled cheese sandwich         1.06  0.60  1.78  0.23  234127.36  147456   
hamburger                       1.03  0.56  1.78  0.22  232550.40  147456   
ice cream                       1.01  0.56  1.67  0.24  228587.52  147968   
pizza                           1.13  0.75  1.53  0.23  225802.24  171520   

                                           
                            max       std  
Class                                      
apple pie                262144  38185.31  
bibimbap                 262144  37043.44  
chicken wings            262144  37728.73  
donuts                   262144  40274.50  
eggs benedict            262144  39054.78  
french fries             262144  38115.24  
grilled cheese sandwich  262144  37467.29  
hamburger                262144  36351.04  
ice cream                262144  38318.79  
pizza                    262144  35858.28

In [12]:
# Visualize image dimensions distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Width distribution
ax = axes[0, 0]
sns.boxplot(data=image_stats_df, x='Class', y='Width', ax=ax)
ax.set_title('Image Width Distribution by Class', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Width (pixels)', fontsize=11)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3)

# Height distribution
ax = axes[0, 1]
sns.boxplot(data=image_stats_df, x='Class', y='Height', ax=ax)
ax.set_title('Image Height Distribution by Class', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Height (pixels)', fontsize=11)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3)

# Aspect ratio distribution
ax = axes[1, 0]
sns.violinplot(data=image_stats_df, x='Class', y='Aspect Ratio', ax=ax)
ax.set_title('Image Aspect Ratio Distribution (Width/Height)', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Aspect Ratio', fontsize=11)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3)

# Area (Width × Height) distribution
ax = axes[1, 1]
sns.histplot(data=image_stats_df, x='Area', hue='Class', kde=True, ax=ax, bins=30)
ax.set_title('Image Area Distribution (all classes)', fontweight='bold')
ax.set_xlabel('Area (pixels²)', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.legend(title='Class', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_image_dimensions_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Image characteristics visualization created and saved")


/var/folders/jr/_vnkd1dx5hz54ytwj94g23sr0000gn/T/ipykernel_93645/3801702522.py:37: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(title='Class', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)


✓ Image characteristics visualization created and saved


/var/folders/jr/_vnkd1dx5hz54ytwj94g23sr0000gn/T/ipykernel_93645/3801702522.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
# Scatter plot: Width vs Height colored by aspect ratio
fig, ax = plt.subplots(figsize=(12, 8))

scatter = ax.scatter(image_stats_df['Width'], image_stats_df['Height'], 
                    c=image_stats_df['Aspect Ratio'], s=50, alpha=0.6, cmap='viridis')
ax.set_xlabel('Width (pixels)', fontsize=12)
ax.set_ylabel('Height (pixels)', fontsize=12)
ax.set_title('Image Dimensions: Width vs Height (colored by Aspect Ratio)', fontweight='bold', fontsize=13)
ax.grid(alpha=0.3)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Aspect Ratio', fontsize=11)

# Add reference lines for square images
ax.plot([0, 3000], [0, 3000], 'r--', alpha=0.3, label='Square (AR=1)')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '05_image_dimensions_scatter.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Dimension scatter plot created and saved")


✓ Dimension scatter plot created and saved


/var/folders/jr/_vnkd1dx5hz54ytwj94g23sr0000gn/T/ipykernel_93645/778954015.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Visual Sample Exploration


In [14]:
# Display sample images from each class (validation split)
validation_dataset = datasets_by_split['validation']

fig, axes = plt.subplots(2, 5, figsize=(16, 8))
axes = axes.flatten()

# Get one sample from each class
for class_id in range(len(class_names)):
    for sample in validation_dataset:
        if sample['label'] == class_id:
            ax = axes[class_id]
            ax.imshow(sample['image'])
            ax.set_title(f"{sample['label_name'].replace('_', ' ').title()}", 
                        fontweight='bold', fontsize=11)
            ax.axis('off')
            break

plt.suptitle('Sample Images from Each Food101 Class (Validation Split)', 
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '06_sample_images_per_class.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Sample images displayed and saved")


/Users/khoatran/Developer/learning-so-deep/venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


✓ Sample images displayed and saved


/var/folders/jr/_vnkd1dx5hz54ytwj94g23sr0000gn/T/ipykernel_93645/3777516183.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# Display multiple samples per class to show variety
n_samples_per_class = 3
fig, axes = plt.subplots(len(class_names), n_samples_per_class, figsize=(12, 16))

for class_id in range(len(class_names)):
    sample_count = 0
    for sample in validation_dataset:
        if sample['label'] == class_id and sample_count < n_samples_per_class:
            ax = axes[class_id, sample_count]
            ax.imshow(sample['image'])
            if sample_count == 0:
                ax.set_ylabel(humanize_class_name(class_names[class_id]), 
                            fontsize=10, fontweight='bold', labelpad=10)
            ax.axis('off')
            sample_count += 1
        
        if sample_count >= n_samples_per_class:
            break

plt.suptitle(f'{n_samples_per_class} Sample Images from Each Food101 Class', 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '07_multiple_samples_per_class.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Multiple samples per class displayed and saved ({n_samples_per_class} images each)")


✓ Multiple samples per class displayed and saved (3 images each)


/var/folders/jr/_vnkd1dx5hz54ytwj94g23sr0000gn/T/ipykernel_93645/4232918078.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Split Composition Analysis


In [16]:
# Analyze split composition
split_composition = []

for split_name, dataset in datasets_by_split.items():
    total = len(dataset)
    split_composition.append({
        'Split': split_name.replace('_', ' ').title(),
        'Count': total,
        'Percentage': (total / sum(len(d) for d in datasets_by_split.values())) * 100
    })

composition_df = pd.DataFrame(split_composition)
print("\nDataset Split Composition:")
print(composition_df.to_string(index=False))

# Pie chart for split distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
colors = sns.color_palette("Set2", len(split_composition))
axes[0].pie(composition_df['Count'], labels=composition_df['Split'], autopct='%1.1f%%',
           colors=colors, startangle=90, textprops={'fontsize': 11, 'weight': 'bold'})
axes[0].set_title('Overall Dataset Split Distribution', fontweight='bold', fontsize=12)

# Bar chart with annotation
ax = axes[1]
bars = ax.bar(composition_df['Split'], composition_df['Count'], color=colors, alpha=0.8)
ax.set_ylabel('Number of Samples', fontsize=11)
ax.set_title('Dataset Splits by Sample Count', fontweight='bold', fontsize=12)
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{int(height)}\n({height/composition_df["Count"].sum()*100:.1f}%)',
           ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '08_split_composition_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Split composition analysis created and saved")



Dataset Split Composition:
         Split  Count  Percentage
Few Shot Train     80    2.877698
  Few Shot Dev    200    7.194245
    Validation   2500   89.928058
✓ Split composition analysis created and saved


/var/folders/jr/_vnkd1dx5hz54ytwj94g23sr0000gn/T/ipykernel_93645/4091868235.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Class-wise Detailed Statistics


In [17]:
# Comprehensive class-wise statistics
class_stats = []

for class_id, class_name in enumerate(class_names):
    for split_name, dataset in datasets_by_split.items():
        count = sum(1 for sample in dataset if sample['label'] == class_id)
        class_stats.append({
            'Class': humanize_class_name(class_name),
            'Split': split_name.replace('_', ' ').title(),
            'Count': count
        })

class_stats_df = pd.DataFrame(class_stats)

# Calculate totals per class
totals_by_class = class_stats_df.groupby('Class')['Count'].sum().sort_values(ascending=False)

print("\nTotal Samples per Class:")
print(totals_by_class)

# Heatmap of class-split combinations
pivot_class_stats = class_stats_df.pivot(index='Class', columns='Split', values='Count').fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pivot_class_stats, annot=True, fmt='d', cmap='YlOrRd', ax=ax, 
           cbar_kws={'label': 'Sample Count'}, linewidths=0.5, linecolor='gray')
ax.set_title('Class Distribution Heatmap Across Splits', fontweight='bold', fontsize=13)
ax.set_xlabel('Dataset Split', fontsize=11)
ax.set_ylabel('Food Class', fontsize=11)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '09_class_distribution_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Class-wise statistics heatmap created and saved")


/Users/khoatran/Developer/learning-so-deep/venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))



Total Samples per Class:
Class
apple pie                  278
bibimbap                   278
chicken wings              278
donuts                     278
eggs benedict              278
french fries               278
grilled cheese sandwich    278
hamburger                  278
ice cream                  278
pizza                      278
Name: Count, dtype: int64
✓ Class-wise statistics heatmap created and saved


/var/folders/jr/_vnkd1dx5hz54ytwj94g23sr0000gn/T/ipykernel_93645/2633973786.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
# Comparison of class popularity
fig, ax = plt.subplots(figsize=(12, 6))

totals_by_class_sorted = totals_by_class.sort_values(ascending=True)
colors = sns.color_palette("husl", len(totals_by_class_sorted))
bars = ax.barh(range(len(totals_by_class_sorted)), totals_by_class_sorted.values, color=colors)

ax.set_yticks(range(len(totals_by_class_sorted)))
ax.set_yticklabels(totals_by_class_sorted.index, fontsize=11)
ax.set_xlabel('Total Number of Samples', fontsize=12)
ax.set_title('Total Samples per Food Class (All Splits Combined)', fontweight='bold', fontsize=13)
ax.grid(axis='x', alpha=0.3)

# Add value labels
for i, (bar, value) in enumerate(zip(bars, totals_by_class_sorted.values)):
    ax.text(value + 5, i, str(int(value)), va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '10_class_popularity_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Class popularity comparison created and saved")


✓ Class popularity comparison created and saved


/var/folders/jr/_vnkd1dx5hz54ytwj94g23sr0000gn/T/ipykernel_93645/788196985.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Summary Statistics and Insights


In [19]:
# Print comprehensive EDA summary
print("="*70)
print("COMPREHENSIVE EDA SUMMARY - FOOD101 SELECTED CLASSES")
print("="*70)

print("\n📊 DATASET OVERVIEW:")
print(f"  • Total Classes Analyzed: {len(class_names)}")
print(f"  • Classes: {', '.join([humanize_class_name(c) for c in class_names])}")
total_samples = sum(len(d) for d in datasets_by_split.values())
print(f"  • Total Samples: {total_samples:,}")

print("\n📈 SPLIT BREAKDOWN:")
for split_name, dataset in datasets_by_split.items():
    count = len(dataset)
    pct = (count / total_samples) * 100
    print(f"  • {split_name.replace('_', ' ').title()}: {count:,} samples ({pct:.1f}%)")

print("\n⚖️  CLASS BALANCE INFO:")
validation_counts = []
for class_id in range(len(class_names)):
    count = sum(1 for s in datasets_by_split['validation'] if s['label'] == class_id)
    validation_counts.append(count)

val_counts_arr = np.array(validation_counts)
print(f"  • Min samples per class: {val_counts_arr.min()}")
print(f"  • Max samples per class: {val_counts_arr.max()}")
print(f"  • Mean samples per class: {val_counts_arr.mean():.1f}")
print(f"  • Std Dev: {val_counts_arr.std():.2f}")
imbalance = (val_counts_arr.max() - val_counts_arr.min()) / val_counts_arr.max()
print(f"  • Imbalance Ratio: {imbalance:.4f} {'✓ Good' if imbalance < 0.2 else '⚠ Moderate' if imbalance < 0.4 else '✗ Poor'}")

print("\n🖼️  IMAGE CHARACTERISTICS:")
print(f"  • Sample Width Range: {image_stats_df['Width'].min():.0f} - {image_stats_df['Width'].max():.0f} pixels")
print(f"  • Sample Height Range: {image_stats_df['Height'].min():.0f} - {image_stats_df['Height'].max():.0f} pixels")
print(f"  • Mean Aspect Ratio: {image_stats_df['Aspect Ratio'].mean():.3f}")
print(f"  • Aspect Ratio Range: {image_stats_df['Aspect Ratio'].min():.3f} - {image_stats_df['Aspect Ratio'].max():.3f}")
print(f"  • Mean Image Area: {image_stats_df['Area'].mean():.0f} pixels²")

print("\n🎯 RECOMMENDATIONS:")
if imbalance < 0.1:
    print("  ✓ Dataset is well-balanced across classes")
    print("  ✓ Standard classification approaches should work well")
else:
    print("  ⚠ Dataset shows moderate class imbalance")
    print("  → Consider techniques like weighted loss or oversampling")

if image_stats_df['Aspect Ratio'].std() > 0.3:
    print("  ⚠ High variation in image aspect ratios detected")
    print("  → Pre-processing may be needed (e.g., padding, resizing strategy)")
else:
    print("  ✓ Images have consistent aspect ratios")

print("\n" + "="*70)


COMPREHENSIVE EDA SUMMARY - FOOD101 SELECTED CLASSES

📊 DATASET OVERVIEW:
  • Total Classes Analyzed: 10
  • Classes: apple pie, bibimbap, chicken wings, donuts, eggs benedict, french fries, grilled cheese sandwich, hamburger, ice cream, pizza
  • Total Samples: 2,780

📈 SPLIT BREAKDOWN:
  • Few Shot Train: 80 samples (2.9%)
  • Few Shot Dev: 200 samples (7.2%)
  • Validation: 2,500 samples (89.9%)

⚖️  CLASS BALANCE INFO:


/Users/khoatran/Developer/learning-so-deep/venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


  • Min samples per class: 250
  • Max samples per class: 250
  • Mean samples per class: 250.0
  • Std Dev: 0.00
  • Imbalance Ratio: 0.0000 ✓ Good

🖼️  IMAGE CHARACTERISTICS:
  • Sample Width Range: 288 - 512 pixels
  • Sample Height Range: 288 - 512 pixels
  • Mean Aspect Ratio: 1.084
  • Aspect Ratio Range: 0.562 - 1.778
  • Mean Image Area: 230222 pixels²

🎯 RECOMMENDATIONS:
  ✓ Dataset is well-balanced across classes
  ✓ Standard classification approaches should work well
  ✓ Images have consistent aspect ratios

